# Evaluate Concept Accuracy from test_samples_evaluation.json

This notebook does the following:
1. Loads the evaluation JSON file.
2. Counts how many images are labeled.
3. Defines labeled images as those where all attribute values are floats.
4. Computes concept accuracy per labeled image and the dataset average concept accuracy.

In [1]:
import json
from pathlib import Path
from statistics import mean

In [2]:
# 1) Load evaluation file
eval_path = Path('checkpoints/cub200_experiment_llama_p20/classification/cub200/test_samples_evaluation.json')

if not eval_path.exists():
    raise FileNotFoundError(f'File not found: {eval_path}')

with eval_path.open('r') as f:
    evaluation_data = json.load(f)

print(f'Loaded file: {eval_path}')
print(f'Total images in file: {len(evaluation_data)}')

Loaded file: checkpoints/cub200_experiment_llama_p20/classification/cub200/test_samples_evaluation.json
Total images in file: 5790


In [3]:
# 2) and 3) Identify labeled images
# Labeled image rule: all attribute values are floats
def is_labeled_image(attrs: dict) -> bool:
    if not isinstance(attrs, dict) or len(attrs) == 0:
        return False
    return all(isinstance(v, float) for v in attrs.values())

labeled_images = {}
unlabeled_images = {}

for image_name, attrs in evaluation_data.items():
    if is_labeled_image(attrs):
        labeled_images[image_name] = attrs
    else:
        unlabeled_images[image_name] = attrs

print(f'Labeled images: {len(labeled_images)}')
print(f'Unlabeled images: {len(unlabeled_images)}')

Labeled images: 100
Unlabeled images: 5690


In [4]:
# 4) Compute concept_accuracy for each labeled image
# concept_accuracy(image) = average of attribute scores for that image
concept_accuracy_by_image = {}

for image_name, attrs in labeled_images.items():
    values = list(attrs.values())
    concept_accuracy_by_image[image_name] = mean(values)

print(f'Computed concept_accuracy for {len(concept_accuracy_by_image)} labeled images.')

Computed concept_accuracy for 100 labeled images.


In [5]:
# Average concept_accuracy across all labeled images
if len(concept_accuracy_by_image) == 0:
    overall_average_concept_accuracy = None
    print('No labeled images found, so overall average cannot be computed.')
else:
    overall_average_concept_accuracy = mean(concept_accuracy_by_image.values())
    print(f'Overall average concept_accuracy (labeled images only): {overall_average_concept_accuracy:.4f}')

Overall average concept_accuracy (labeled images only): 0.9098
